# Florence-2 Analyzer — Backend
Zellen **der Reihe nach** ausführen. Nach Zelle 5 erscheint die öffentliche URL.

In [ ]:
import subprocess, sys, os

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'transformers', 'timm', 'einops', 'pillow',
    'fastapi', 'uvicorn[standard]', 'python-multipart',
])

if not os.path.exists('./cloudflared'):
    subprocess.run(['wget', '-q', '-O', 'cloudflared',
        'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64'])
    subprocess.run(['chmod', '+x', './cloudflared'])

print('✓ done — KERNEL NEU STARTEN bevor Zelle 2 läuft!')

In [ ]:
import os

# Fix 1: TensorFlow crasht wegen NumPy 2.x
# muss als env-var gesetzt sein BEVOR transformers importiert wird
os.environ['USE_TF']  = '0'
os.environ['USE_JAX'] = '0'

# Fix 2: flash_attn ist kaputt installiert — check_imports patchen
# damit transformers nicht abbricht bevor das Modell geladen wird
import transformers.dynamic_module_utils as _dmu
_orig_check = _dmu.check_imports
def _patched_check(filename):
    try:
        return _orig_check(filename)
    except ImportError as e:
        if 'flash_attn' in str(e):
            return _dmu.get_relative_imports(filename)
        raise
_dmu.check_imports = _patched_check

print('✓ patches applied')

In [ ]:
import torch, io, threading, subprocess, re, time, requests
from PIL import Image
from transformers import AutoProcessor, AutoModelForCausalLM

device      = 'cuda:0' if torch.cuda.is_available() else 'cpu'
torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32

print(f'Torch {torch.__version__}')
print(f'CUDA:  {torch.cuda.is_available()}')
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f'GPU:   {p.name}')
    print(f'VRAM:  {p.total_memory/1e9:.1f} GB')

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    'microsoft/Florence-2-large',
    torch_dtype=torch_dtype,
    trust_remote_code=True
).to(device)

processor = AutoProcessor.from_pretrained(
    'microsoft/Florence-2-large',
    trust_remote_code=True
)

# Smoke-test mit HF-Beispielbild
_url = 'https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/transformers/tasks/car.jpg'
_img = Image.open(requests.get(_url, stream=True).raw)
_inp = processor(text='<CAPTION>', images=_img, return_tensors='pt').to(device, torch_dtype)
with torch.no_grad():
    _ids = model.generate(input_ids=_inp['input_ids'],
                          pixel_values=_inp['pixel_values'], max_new_tokens=20)
print('✓ model ok:', processor.batch_decode(_ids, skip_special_tokens=False)[0][:80])

In [ ]:
from fastapi import FastAPI, UploadFile, File, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import JSONResponse
import uvicorn


def run_task(image, task, text_input=''):
    inputs = processor(
        text=task + text_input, images=image, return_tensors='pt'
    ).to(device, torch_dtype)
    with torch.no_grad():
        ids = model.generate(
            input_ids=inputs['input_ids'],
            pixel_values=inputs['pixel_values'],
            max_new_tokens=1024,
            num_beams=3,
            do_sample=False
        )
    raw = processor.batch_decode(ids, skip_special_tokens=False)[0]
    return processor.post_process_generation(
        raw, task=task, image_size=(image.width, image.height)
    )


app = FastAPI()
app.add_middleware(CORSMiddleware,
    allow_origins=['*'], allow_methods=['*'], allow_headers=['*'])


@app.get('/health')
def health():
    return {'status': 'ok', 'device': device}


@app.post('/analyze')
async def analyze(file: UploadFile = File(...)):
    try:
        image = Image.open(io.BytesIO(await file.read())).convert('RGB')
        result = {'imageWidth': image.width, 'imageHeight': image.height}

        # --- Tasks ohne Text-Input ---
        for key, task in [
            ('caption',          '<DETAILED_CAPTION>'),
            ('caption2',         '<MORE_DETAILED_CAPTION>'),
            ('objects',          '<OD>'),
            ('regions',          '<DENSE_REGION_CAPTION>'),
            ('proposals',        '<REGION_PROPOSAL>'),
            ('ocr',              '<OCR>'),
            ('ocr_regions',      '<OCR_WITH_REGION>'),
        ]:
            try:
                result[key] = run_task(image, task)
            except Exception as e:
                result[key] = {'error': str(e)}

        # --- Phrase Grounding: braucht Caption als Input ---
        try:
            caption_text = result.get('caption', {}).get('<DETAILED_CAPTION>', '')
            if caption_text:
                result['phrases'] = run_task(
                    image, '<CAPTION_TO_PHRASE_GROUNDING>', caption_text
                )
        except Exception as e:
            result['phrases'] = {'error': str(e)}

        return JSONResponse(content=result)
    except Exception as e:
        raise HTTPException(status_code=500, detail=str(e))


print('✓ API defined')

In [ ]:
threading.Thread(
    target=lambda: uvicorn.run(app, host='0.0.0.0', port=8000, log_level='warning'),
    daemon=True
).start()
time.sleep(2)
print('✓ server on :8000')

_t = subprocess.Popen(
    ['./cloudflared', 'tunnel', '--url', 'http://localhost:8000'],
    stderr=subprocess.PIPE, stdout=subprocess.PIPE, text=True
)
for _line in _t.stderr:
    _m = re.search(r'https://[a-z0-9\-]+\.trycloudflare\.com', _line)
    if _m:
        print()
        print('=' * 55)
        print(f'  URL : {_m.group()}')
        print(f'  → in index.html eintragen')
        print('=' * 55)
        break